# 🎨 SVG Renderer · Fine-tuning Notebook

Fine-tune **DeepSeek Coder 1.3B** to map `chart_spec (JSON) → SVG code` using
QLoRA with [Unsloth](https://github.com/unslothai/unsloth).

- **Dataset**: [`DanielRegaladoCardoso/svg-chart-render-v1`](https://huggingface.co/datasets/DanielRegaladoCardoso/svg-chart-render-v1) — ~25 k `(chart_spec, svg)` pairs
- **Base model**: [`unsloth/deepseek-coder-1.3b-instruct-bnb-4bit`](https://huggingface.co/unsloth/deepseek-coder-1.3b-instruct-bnb-4bit)
- **Hardware**: Colab **T4 free tier** ✅ (15 GB VRAM)
- **Expected time**: ~1–2 h for 1 epoch

> [`https://github.com/DanielRegaladoUMiami/sql-agent-llmops`](https://github.com/DanielRegaladoUMiami/sql-agent-llmops)


## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install Unsloth + deps

In [ ]:
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets


## 3 · Log in to HuggingFace (for pushing the adapter at the end)

In [ ]:
from huggingface_hub import login
login()  # paste your HF token with `write` scope


## 4 · Load the base model in 4-bit

This cell tries the pre-quantized Unsloth checkpoint first and falls back to
the official HF checkpoint (with bitsandbytes 4-bit on the fly) if the Unsloth
repo is missing or incompatible. This guards against upstream repo churn.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

PRIMARY  = "unsloth/deepseek-coder-1.3b-instruct-bnb-4bit"
FALLBACK = "deepseek-ai/deepseek-coder-1.3b-instruct"

def load_with_fallback(primary, fallback, max_seq_length=MAX_SEQ_LEN):
    try:
        print(f"Trying primary: {primary}")
        return FastLanguageModel.from_pretrained(
            model_name=primary, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )
    except Exception as e:
        print(f"⚠️  Primary failed ({type(e).__name__}). Falling back to {fallback}.")
        return FastLanguageModel.from_pretrained(
            model_name=fallback, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )

model, tokenizer = load_with_fallback(PRIMARY, FALLBACK)


## 5 · Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
    use_rslora         = False,
)


## 6 · Load dataset from HuggingFace

In [ ]:
from datasets import load_dataset
import json

ds = load_dataset("DanielRegaladoCardoso/svg-chart-render-v1")
print(ds)
print()
print("Sample chart_spec:", ds["train"][0]["chart_spec"][:200], "…")
print("Sample svg_code :", ds["train"][0]["svg_code"][:200], "…")


## 7 · Format as chat messages for SFT

Chart_spec / metadata are stored as JSON strings in the dataset — we parse them
and wrap in a standard chat template. Only the `assistant` turn contributes
to loss (via SFTTrainer's built-in completion-only masking).

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert data-visualization engineer. "
    "Given a chart specification as JSON, produce a clean, valid inline SVG."
)

def to_chat(row):
    spec = row["chart_spec"]  # already a JSON string
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Render this chart spec as SVG:\n\n{spec}"},
        {"role": "assistant", "content": row["svg_code"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_ds = ds["train"].map(to_chat, remove_columns=ds["train"].column_names)
eval_ds  = ds["validation"].map(to_chat, remove_columns=ds["validation"].column_names)
print(train_ds)
print("\n--- sample text (truncated) ---")
print(train_ds[0]["text"][:800])


## 7.5 · Filter rows that exceed MAX_SEQ_LEN

Truncation is silent and causes the model to learn SVGs without seeing the
full chart_spec (or vice-versa). We filter upfront.

In [ ]:
def fits(row):
    return len(tokenizer.encode(row["text"], add_special_tokens=False)) <= MAX_SEQ_LEN

_before_train = len(train_ds)
_before_eval  = len(eval_ds)

train_ds = train_ds.filter(fits, num_proc=2)
eval_ds  = eval_ds.filter(fits,  num_proc=2)

print(f"Train: kept {len(train_ds):,} / {_before_train:,} "
      f"({100*len(train_ds)/max(1,_before_train):.1f}%)")
print(f"Eval : kept {len(eval_ds):,} / {_before_eval:,} "
      f"({100*len(eval_ds)/max(1,_before_eval):.1f}%)")


## 8 · Trainer config

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = "svg_renderer_adapter",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.03,
    num_train_epochs            = 1,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    save_steps                  = 500,
    save_total_limit            = 2,
    optim                       = "adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    report_to                   = "none",
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds.select(range(min(500, len(eval_ds)))),
    args           = training_args,
)


## 9 · Train 🚀

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)


## 10 · Quick inference test

In [ ]:
FastLanguageModel.for_inference(model)

import json as _json
sample = ds["test"][0]
spec   = sample["chart_spec"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": f"Render this chart spec as SVG:\n\n{spec}"},
]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out = model.generate(input_ids, max_new_tokens=2000, do_sample=False, temperature=0)
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)[:1500])


## 11 · Push LoRA adapter to HuggingFace

In [ ]:
model.push_to_hub("DanielRegaladoCardoso/svg-renderer-deepseek-coder-1.3b-lora",
                  tokenizer=tokenizer, save_method="lora")


## ✅ Done

Your adapter is at:
**https://huggingface.co/DanielRegaladoCardoso/svg-renderer-deepseek-coder-1.3b-lora**

To use it in production:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "DanielRegaladoCardoso/svg-renderer-deepseek-coder-1.3b-lora",
    max_seq_length = 2048, load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
```
